# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassan-tech-pro/ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# We compute an explicit heuristic Action Score to rank candidate items for optimization or resource reallocation. High impressions with low conversion or click-through rates trigger an optimization flag, whereas low resource utilization combined with high baseline memory triggers a compression candidate flag.
import pandas as pd
import numpy as np

# Verify reason code logic against a small sample frame
def assign_reason_code(row):
    if row.get('gsc_impressions', 0) > 1000 and (row.get('gsc_clicks', 0) / max(row.get('gsc_impressions', 1), 1)) < 0.01:
        return 'RC_HIGH_IMP_LOW_CTR'
    elif row.get('sessions_direct', 0) < 50 and row.get('ga4_total_engagement_sec', 0) < 100:
        return 'RC_HIGH_MEM_LOW_UTIL'
    return 'RC_STABLE_BASELINE'

print("Reason code function initialized successfully.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Reason code function initialized successfully.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# We build a composite priority score normalized between $0.0$ and $1.0$. The queue is sorted in descending order by action_score. The resulting dataset is written directly to work/outputs/baseline_action_score.csv.
import pandas as pd
import duckdb
import os
from huggingface_hub import HfFileSystem
from google.colab import userdata

# Retrieve active token
try:
    hf_token = userdata.get('HN_TOKEN')
except Exception:
    # If not in Colab Secrets, replace with your active token string directly:
    hf_token = "YOUR_NEW_ACTIVE_HF_TOKEN"

# 1. Access dataset with active token
fs = HfFileSystem(token=hf_token)
repo_path = "datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
files = fs.glob(repo_path)

# 2. Read parquet files into DataFrame
dfs = []
for file in files:
    with fs.open(file, 'rb') as f:
        dfs.append(pd.read_parquet(f))

df_raw = pd.concat(dfs, ignore_index=True)

# 3. Filter available data and calculate action scores
df_filtered = df_raw[df_raw['gsc_data_available'] == True].copy()
df_filtered['raw_score'] = (df_filtered['gsc_impressions'] * 0.6) + (df_filtered['gsc_clicks'] * 0.4)

max_s = df_filtered['raw_score'].max() if df_filtered['raw_score'].max() > 0 else 1
df_filtered['action_score'] = (df_filtered['raw_score'] / max_s).round(4)

# 4. Rank and write output CSV
df_ranked = df_filtered.sort_values(by='action_score', ascending=False).reset_index(drop=True)
df_ranked['rank'] = df_ranked.index + 1

out_cols = ['report_date', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'sessions_ai', 'raw_score', 'action_score', 'rank']

os.makedirs('work/outputs', exist_ok=True)
df_ranked[out_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Successfully generated baseline queue with {len(df_ranked)} records in work/outputs/baseline_action_score.csv")
display(df_ranked[out_cols].head(5))

Successfully generated baseline queue with 3611061 records in work/outputs/baseline_action_score.csv


,report_date,content_hash_id,gsc_impressions,gsc_clicks,sessions_ai,raw_score,action_score,rank
0,2026-03-28,content_44f34c0a90047651,40084,1,0.0,24050.8,1.0000,1
1,2026-03-29,content_eadb33b5df496f4a,39305,252,0.0,23683.8,0.9847,2
2,2026-03-04,content_34a70fea29d15f24,39003,2,NaN,23402.6,0.9730,3
3,2026-03-28,content_eadb33b5df496f4a,38436,271,0.0,23170.0,0.9634,4
4,2026-03-04,content_945d6ff91386c817,37368,0,NaN,22420.8,0.9322,5


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd
import os

# Define output path
output_path = 'work/outputs/baseline_action_score.csv'

if os.path.exists(output_path):
    df_out = pd.read_csv(output_path)

    print("=== Baseline Queue Generation Summary ===")
    print(f"Total Rows: {len(df_out)}")
    print(f"Columns: {list(df_out.columns)}")
    print("\n--- Action Score Statistics ---")
    print(df_out['action_score'].describe())

    print("\n--- Top 5 Ranked Articles ---")
    display(df_out.head(5))
else:
    print(f"Error: Output file not found at {output_path}")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


=== Baseline Queue Generation Summary ===
Total Rows: 3611061
Columns: ['report_date', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'sessions_ai', 'raw_score', 'action_score', 'rank']

--- Action Score Statistics ---
count    3.611061e+06
mean     1.931388e-03
std      6.249109e-03
min      0.000000e+00
25%      1.000000e-04
50%      4.000000e-04
75%      1.500000e-03
max      1.000000e+00
Name: action_score, dtype: float64

--- Top 5 Ranked Articles ---


,report_date,content_hash_id,gsc_impressions,gsc_clicks,sessions_ai,raw_score,action_score,rank
0,2026-03-28,content_44f34c0a90047651,40084,1,0.0,24050.8,1.0000,1
1,2026-03-29,content_eadb33b5df496f4a,39305,252,0.0,23683.8,0.9847,2
2,2026-03-04,content_34a70fea29d15f24,39003,2,NaN,23402.6,0.9730,3
3,2026-03-28,content_eadb33b5df496f4a,38436,271,0.0,23170.0,0.9634,4
4,2026-03-04,content_945d6ff91386c817,37368,0,NaN,22420.8,0.9322,5


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
import numpy as np

# Load ranked baseline dataset
df_eval = pd.read_csv('work/outputs/baseline_action_score.csv')

# Calculate performance quantiles
df_eval['priority_tier'] = pd.qcut(
    df_eval['action_score'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

# Priority distribution summary
tier_summary = df_eval.groupby('priority_tier').agg(
    count=('content_hash_id', 'count'),
    mean_impressions=('gsc_impressions', 'mean'),
    mean_clicks=('gsc_clicks', 'mean'),
    mean_action_score=('action_score', 'mean')
).reset_index()

print("=== Performance Tiers Summary ===")
display(tier_summary)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


/tmp/ipykernel_2015/1873944965.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tier_summary = df_eval.groupby('priority_tier').agg(


=== Performance Tiers Summary ===


,priority_tier,count,mean_impressions,mean_clicks,mean_action_score
0,Low,1207867,2.757919,0.007031,0.000048
1,Medium,1205176,17.888868,0.045174,0.000438
2,High,1198018,213.491914,0.633460,0.005333


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.